In [51]:
!pip install nltk

In [52]:
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from nltk.tokenize import word_tokenize as word_tokenize


In [53]:
Document = "My name is Yogendra. I am a B.Tech 3rd year student from the Department of Bioscience and Bioengineering at IIT Jodhpur. I am very passionate about Deep Learning and Artificial Intelligence. I love to work on new projects and learn new things. I am also interested to manage the events and activities.I always try to give my best in whatever I do. I am Internship Coordinator of training and placement cell of IIT Jodhpur."

In [54]:
tokens = word_tokenize(Document.lower())

In [55]:
tokens

['my',
 'name',
 'is',
 'yogendra',
 '.',
 'i',
 'am',
 'a',
 'b.tech',
 '3rd',
 'year',
 'student',
 'from',
 'the',
 'department',
 'of',
 'bioscience',
 'and',
 'bioengineering',
 'at',
 'iit',
 'jodhpur',
 '.',
 'i',
 'am',
 'very',
 'passionate',
 'about',
 'deep',
 'learning',
 'and',
 'artificial',
 'intelligence',
 '.',
 'i',
 'love',
 'to',
 'work',
 'on',
 'new',
 'projects',
 'and',
 'learn',
 'new',
 'things',
 '.',
 'i',
 'am',
 'also',
 'interested',
 'to',
 'manage',
 'the',
 'events',
 'and',
 'activities.i',
 'always',
 'try',
 'to',
 'give',
 'my',
 'best',
 'in',
 'whatever',
 'i',
 'do',
 '.',
 'i',
 'am',
 'internship',
 'coordinator',
 'of',
 'training',
 'and',
 'placement',
 'cell',
 'of',
 'iit',
 'jodhpur',
 '.']

In [56]:
from collections import Counter
vocab = {'<unk>': 0}
for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab)

In [57]:
len(vocab)

55

In [58]:
input_sentences = Document.split('/n')

In [59]:
def text_to_indices(sentence,vocab):
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])
            
    return numerical_sentence

In [60]:
input_numerical_sentences = []
for sentence in input_sentences:
    input_numerical_sentences.append(text_to_indices(word_tokenize(sentence),vocab))


In [61]:
len(input_numerical_sentences)


1

In [62]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [63]:
len(training_sequence)

79

In [64]:
training_sequence[:5]

[[0, 2], [0, 2, 3], [0, 2, 3, 0], [0, 2, 3, 0, 5], [0, 2, 3, 0, 5, 0]]

In [65]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

80

In [66]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [67]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [68]:
padded_training_sequence

tensor([[ 0,  0,  0,  ...,  0,  0,  2],
        [ 0,  0,  0,  ...,  0,  2,  3],
        [ 0,  0,  0,  ...,  2,  3,  0],
        ...,
        [ 0,  0,  0,  ..., 54, 16,  0],
        [ 0,  0,  2,  ..., 16,  0,  0],
        [ 0,  2,  3,  ...,  0,  0,  5]])

In [69]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:, 1:]

In [70]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):                                                           
    return self.X.shape[0]                                                                                                                                                                                                                                             

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [71]:
dataset = CustomDataset(X,y)

In [72]:
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

In [73]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(intermediate_hidden_states)
    return output

In [74]:
model = LSTMModel(len(vocab))

In [75]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [76]:
device

device(type='cuda')

In [77]:
model.to(device)

LSTMModel(
  (embedding): Embedding(55, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=55, bias=True)
)

In [78]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [79]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output.view(-1, output.shape[-1]), batch_y.view(-1))

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 30.6003
Epoch: 2, Loss: 18.3266
Epoch: 3, Loss: 16.1392
Epoch: 4, Loss: 14.6861
Epoch: 5, Loss: 12.9772
Epoch: 6, Loss: 11.1173
Epoch: 7, Loss: 8.9928
Epoch: 8, Loss: 7.0462
Epoch: 9, Loss: 5.4721
Epoch: 10, Loss: 4.3152
Epoch: 11, Loss: 3.4646
Epoch: 12, Loss: 2.8371
Epoch: 13, Loss: 2.3719
Epoch: 14, Loss: 2.0217
Epoch: 15, Loss: 1.7443
Epoch: 16, Loss: 1.5437
Epoch: 17, Loss: 1.3841
Epoch: 18, Loss: 1.2774
Epoch: 19, Loss: 1.1818
Epoch: 20, Loss: 1.0961
Epoch: 21, Loss: 1.0325
Epoch: 22, Loss: 0.9752
Epoch: 23, Loss: 0.9386
Epoch: 24, Loss: 0.8980
Epoch: 25, Loss: 0.8779
Epoch: 26, Loss: 0.8742
Epoch: 27, Loss: 0.8718
Epoch: 28, Loss: 0.8302
Epoch: 29, Loss: 0.7984
Epoch: 30, Loss: 0.7722
Epoch: 31, Loss: 0.7526
Epoch: 32, Loss: 0.7415
Epoch: 33, Loss: 0.7260
Epoch: 34, Loss: 0.7218
Epoch: 35, Loss: 0.7128
Epoch: 36, Loss: 0.7051
Epoch: 37, Loss: 0.8303
Epoch: 38, Loss: 1.0956
Epoch: 39, Loss: 0.9418
Epoch: 40, Loss: 0.8351
Epoch: 41, Loss: 0.7832
Epoch: 42, Loss: 0.

In [80]:
def prediction(model, vocab, text, max_len=61):
    model.eval()
    device = next(model.parameters()).device  # GPU

    # tokenize
    tokenized_text = word_tokenize(text.lower())

    # text -> numerical indices
    numerical_text = text_to_indices(tokenized_text, vocab)

    # left padding + move to GPU
    padded_text = torch.tensor(
        [0] * (max_len - len(numerical_text)) + numerical_text,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    # inference
    with torch.no_grad():
        output = model(padded_text)

    # predicted index
    _, index = torch.max(output[0, -1, :], dim=0)
    index = index.item()

    # reverse vocab (index → word)
    idx_to_word = {v: k for k, v in vocab.items()}

    # merge prediction with input text
    return text + " " + idx_to_word.get(index, "<unk>")


In [81]:
prediction(model, vocab, "I am very passionate")

'I am very passionate <unk>'

In [ ]:
import time

num_tokens = 5
input_text = "I am very passionate"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)


I am very passionate <unk>
I am very passionate <unk> <unk>
I am very passionate <unk> <unk> <unk>
I am very passionate <unk> <unk> <unk> <unk>
I am very passionate <unk> <unk> <unk> <unk> <unk>
I am very passionate <unk> <unk> <unk> <unk> <unk> <unk>
I am very passionate <unk> <unk> <unk> <unk> <unk> <unk> <unk>
I am very passionate <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
I am very passionate <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
I am very passionate <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
